In [1]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host='localhost',
    database='f1_predictor',
    user='julissasu',
    password=''
)

df = pd.read_sql("""
    SELECT 
        r.year, r.round, r.race_name, r.circuit_id, r.circuit_name, r.country, r.date,
        d.driver_id, d.driver_number, d.driver_code, d.driver_forename, d.driver_surname, d.driver_full_name,
        t.team_id, t.team_name,
        rr.grid_position, rr.finish_position, rr.position_text, rr.points, 
        rr.laps_completed, rr.status, rr.time, rr.finished, rr.dnf
    FROM race_results rr
    JOIN races r ON rr.race_id = r.race_id
    JOIN drivers d ON rr.driver_id = d.driver_id
    JOIN teams t ON rr.team_id = t.team_id
    WHERE r.year >= 2010
    ORDER BY r.year, r.round, rr.finish_position
""", conn)

conn.close()

print(f"Loaded {len(df)} rows from database")
print(f"Years: {df['year'].min()} to {df['year'].max()}")
print(f"Unique races: {df['race_name'].nunique()}")
print(f"Drivers: {df['driver_full_name'].nunique()}")
print(f"Teams: {df['team_name'].nunique()}")

# Check for missing values
print(f"\nMissing values:")
print(df.isnull().sum())

Loaded 6911 rows from database
Years: 2010 to 2025
Unique races: 40
Drivers: 83
Teams: 23

Missing values:
year                   0
round                  0
race_name              0
circuit_id             0
circuit_name           0
country                0
date                   0
driver_id              0
driver_number        739
driver_code            0
driver_forename        0
driver_surname         0
driver_full_name       0
team_id                0
team_name              0
grid_position          0
finish_position        0
position_text          0
points                 0
laps_completed         0
status                 0
time                2951
finished               0
dnf                    0
dtype: int64


/var/folders/dy/wqpvtj0n01j_f27wlwcbmm_c0000gn/T/ipykernel_6699/197511518.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("""


In [2]:
# Basic features
df_ml = df.copy()

# Feature 1: Driver win rate
driver_wins = df[df['finish_position'] == 1].groupby('driver_full_name').size()
driver_races = df.groupby('driver_full_name').size()
df_ml['driver_win_rate'] = df_ml['driver_full_name'].map(
    driver_wins / driver_races * 100
).fillna(0)

# Feature 2: Team average finish
team_avg = df.groupby('team_name')['finish_position'].mean()
df_ml['team_avg_finish'] = df_ml['team_name'].map(team_avg)

# Feature 3: Recent form
df_ml = df_ml.sort_values(['driver_full_name', 'year', 'round'])
df_ml['driver_recent_form'] = df_ml.groupby('driver_full_name')['finish_position'].transform(
    lambda x: x.rolling(3, min_periods=1).mean()
)

# Target
df_ml['won_race'] = (df_ml['finish_position'] == 1).astype(int)

print("Basic features: driver_win_rate, team_avg_finish, driver_recent_form, grid_position")

Basic features: driver_win_rate, team_avg_finish, driver_recent_form, grid_position


In [3]:
# High priority features

# Feature 4: Driver average finish
driver_avg = df.groupby('driver_full_name')['finish_position'].mean()
df_ml['driver_avg_finish'] = df_ml['driver_full_name'].map(driver_avg)

# Feature 5: Driver podium rate
driver_podiums = df[df['finish_position'] <= 3].groupby('driver_full_name').size()
df_ml['driver_podium_rate'] = df_ml['driver_full_name'].map(
    driver_podiums / driver_races * 100
).fillna(0)

# Feature 6: Circuit-specific performance
circuit_driver_avg = df.groupby(['race_name', 'driver_full_name'])['finish_position'].mean()
df_ml['circuit_driver_performance'] = df_ml.apply(
    lambda row: circuit_driver_avg.get((row['race_name'], row['driver_full_name']), row['driver_avg_finish']),
    axis=1
)

# Feature 7: Qualifying position delta
df_ml['qualifying_position_delta'] = df_ml['grid_position'] - df_ml['finish_position']

print("Advanced features: driver_avg_finish, driver_podium_rate, circuit_driver_performance, qualifying_position_delta")

Advanced features: driver_avg_finish, driver_podium_rate, circuit_driver_performance, qualifying_position_delta


In [4]:
# Weather proxy features from race status
df_ml['wet_race'] = df_ml['status'].str.contains('weather|rain|wet', case=False, na=False).astype(int)

# Driver wet weather skill
wet_races = df[df['status'].str.contains('weather|rain|wet', case=False, na=False)]
driver_wet_skill = wet_races.groupby('driver_full_name')['finish_position'].mean()
df_ml['driver_wet_weather_skill'] = df_ml['driver_full_name'].map(driver_wet_skill)

# Fill NaN for drivers with no wet races
df_ml['driver_wet_weather_skill'] = df_ml['driver_wet_weather_skill'].fillna(df_ml['driver_avg_finish'])

print("Weather features added: wet_race, driver_wet_weather_skill")

Weather features added: wet_race, driver_wet_weather_skill


In [5]:
# Train model
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Prepare features
feature_cols = [
    'driver_win_rate', 'team_avg_finish', 'driver_recent_form', 'grid_position',
    'driver_avg_finish', 'driver_podium_rate', 'circuit_driver_performance', 
    'qualifying_position_delta', 'wet_race', 'driver_wet_weather_skill'
]

X = df_ml[feature_cols].copy().fillna(df_ml[feature_cols].mean())
y = df_ml['won_race'].copy()


# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {len(X_train)} races")
print(f"Test set: {len(X_test)} races")
print(f"Wins in test set: {y_test.sum()}")

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
print("\nModel trained")

# Evaluate
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\nAccuracy: {accuracy*100:.2f}%")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Did Not Win', 'Won']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(f"True Negatives: {cm[0][0]}")
print(f"False Positives: {cm[0][1]}")
print(f"False Negatives: {cm[1][0]}")
print(f"True Positives: {cm[1][1]}")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importance:")
for idx, row in feature_importance.iterrows():
    print(f"{row['feature']:30s} {row['importance']*100:5.1f}%")

Training set: 5528 races
Test set: 1383 races
Wins in test set: 66

Model trained

Accuracy: 99.06%

Classification Report:
              precision    recall  f1-score   support

 Did Not Win       0.99      1.00      1.00      1317
         Won       1.00      0.80      0.89        66

    accuracy                           0.99      1383
   macro avg       1.00      0.90      0.94      1383
weighted avg       0.99      0.99      0.99      1383


Confusion Matrix:
True Negatives: 1317
False Positives: 0
False Negatives: 13
True Positives: 53

Feature Importance:
grid_position                   26.4%
qualifying_position_delta       25.8%
driver_recent_form              24.6%
circuit_driver_performance       7.6%
driver_win_rate                  4.0%
driver_podium_rate               3.3%
driver_avg_finish                3.2%
driver_wet_weather_skill         3.1%
team_avg_finish                  2.1%
wet_race                         0.0%


In [6]:
print("="*60)
print("MODEL COMPARISON: v1.0 vs v2.0")
print("="*60)

wins_correct = ((y_test == 1) & (y_pred == 1)).sum()
win_detection_v2 = wins_correct / y_test.sum() * 100 if y_test.sum() > 0 else 0

print("\nv1.0 (Sample):")
print("  Dataset: 1,359 rows (2022-2024)")
print("  Features: 4")
print("  Accuracy: 95.96%")
print("  Win Detection: 64% (9/14)")

print(f"\nv2.0 (Full Database):")
print(f"  Dataset: {len(df_ml)} rows (2010-2025)")
print(f"  Features: {len(feature_cols)}")
print(f"  Accuracy: {accuracy*100:.2f}%")
print(f"  Win Detection: {win_detection_v2:.1f}% ({wins_correct}/{y_test.sum()})")

print(f"\nImprovement: +{accuracy*100 - 95.96:.2f}% accuracy, +{win_detection_v2 - 64:.1f}% win detection")
print("="*60)

MODEL COMPARISON: v1.0 vs v2.0

v1.0 (Sample):
  Dataset: 1,359 rows (2022-2024)
  Features: 4
  Accuracy: 95.96%
  Win Detection: 64% (9/14)

v2.0 (Full Database):
  Dataset: 6911 rows (2010-2025)
  Features: 10
  Accuracy: 99.06%
  Win Detection: 80.3% (53/66)

Improvement: +3.10% accuracy, +16.3% win detection


In [7]:
# Prediction function
def predict_v2(driver_win_rate, team_avg_finish, driver_recent_form, grid_position, 
               driver_avg_finish, driver_podium_rate, circuit_perf, qual_delta, wet_race, wet_skill):
    input_data = pd.DataFrame([{
        'driver_win_rate': driver_win_rate, 'team_avg_finish': team_avg_finish,
        'driver_recent_form': driver_recent_form, 'grid_position': grid_position,
        'driver_avg_finish': driver_avg_finish, 'driver_podium_rate': driver_podium_rate,
        'circuit_driver_performance': circuit_perf, 'qualifying_position_delta': qual_delta,
        'wet_race': wet_race, 'driver_wet_weather_skill': wet_skill
    }])
    prob = model.predict_proba(input_data)[0][1]
    return f"{prob*100:.1f}%"

print("Test Predictions:\n")
print(f"Verstappen (pole, dry): {predict_v2(63.2, 4.9, 1.5, 1, 6.5, 50, 3.0, 0, 0, 8.0)}")
print(f"Leclerc (P5, dry): {predict_v2(8.8, 6.5, 5.0, 5, 7.5, 25, 7.0, -2, 0, 9.0)}")
print(f"Backmarker (P18, dry): {predict_v2(0.0, 15.0, 16.0, 18, 14.0, 2, 16.0, -2, 0, 15.0)}")

Test Predictions:

Verstappen (pole, dry): 92.5%
Leclerc (P5, dry): 0.4%
Backmarker (P18, dry): 0.0%


In [8]:
import pickle
import json
import os
from datetime import datetime

os.makedirs('../models/v2.0', exist_ok=True)

with open('../models/v2.0/f1_winner_model_v2.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('../models/v2.0/model_features.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)

model_info = {
    'version': '2.0',
    'created_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'dataset': 'Full database (2010-2025)',
    'num_samples': len(X),
    'features': feature_cols,
    'accuracy': f"{accuracy*100:.2f}%",
    'model_type': 'RandomForestClassifier',
    'hyperparameters': {'n_estimators': 100, 'max_depth': 10, 'random_state': 42},
    'improvements_over_v1': {
        'data_increase': f'+{len(df_ml)-1359} rows',
        'feature_increase': f'+{len(feature_cols)-4} features',
        'accuracy_gain': f'+{accuracy*100 - 95.96:.2f}%'
    }
}

with open('../models/v2.0/model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)

print("Model v2.0 saved")

Model v2.0 saved
 Accuracy: 99.06%
